In [135]:
import pandas as pd
from pathlib import Path
project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
file_path = project / "data" / "raw" / "heart_failure_clinical_raw.csv"
data = pd.read_csv(file_path)
print("Dataset shape:", data.shape)
print(data.head())

Dataset shape: (918, 12)
   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  


In [136]:
print("Missing values in each column:")
print(data.isnull().sum())

Missing values in each column:
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64


In [137]:
print("Zero values in each column:")
print((data == 0).sum())

Zero values in each column:
Age                 0
Sex                 0
ChestPainType       0
RestingBP           1
Cholesterol       172
FastingBS         704
RestingECG          0
MaxHR               0
ExerciseAngina      0
Oldpeak           368
ST_Slope            0
HeartDisease      410
dtype: int64


In [138]:
print("Statistical summary:")
print(data.describe())

Statistical summary:
              Age   RestingBP  Cholesterol   FastingBS       MaxHR  \
count  918.000000  918.000000   918.000000  918.000000  918.000000   
mean    53.510893  132.396514   198.799564    0.233115  136.809368   
std      9.432617   18.514154   109.384145    0.423046   25.460334   
min     28.000000    0.000000     0.000000    0.000000   60.000000   
25%     47.000000  120.000000   173.250000    0.000000  120.000000   
50%     54.000000  130.000000   223.000000    0.000000  138.000000   
75%     60.000000  140.000000   267.000000    0.000000  156.000000   
max     77.000000  200.000000   603.000000    1.000000  202.000000   

          Oldpeak  HeartDisease  
count  918.000000    918.000000  
mean     0.887364      0.553377  
std      1.066570      0.497414  
min     -2.600000      0.000000  
25%      0.000000      0.000000  
50%      0.600000      1.000000  
75%      1.500000      1.000000  
max      6.200000      1.000000  


In [139]:
import sys
from pathlib import Path
project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
sys.path.append(str(project))

In [140]:
from src.analyzer import CardioRiskProfiler
file_path = project / "data" / "processed" / "clean_heart_failure.csv"
p = CardioRiskProfiler()
p.load_from_csv(file_path)
print("Oldpeak cohen's disease",p.compute_cohens_d('Oldpeak'))
print("MaxHR Cohen's d:", p.compute_cohens_d('MaxHR'))
print("Age Cohen's d:", p.compute_cohens_d('Age'))


Oldpeak cohen's disease 0.8863573136001801
MaxHR Cohen's d: -0.8805208867307898
Age Cohen's d: 0.5905569689898478


In [141]:
for biomarker in ['Oldpeak', 'MaxHR', 'Age']:
    result = p.compute_cohens_d(biomarker)
    print(biomarker, ":", result)

Oldpeak : 0.8863573136001801
MaxHR : -0.8805208867307898
Age : 0.5905569689898478


In [142]:
print("Verification:", p.verify_cohens_d(biomarker))

Verification: 0.5905569689898478


In [143]:
print(p.check_cohens_d())

True


In [144]:
for biomarker in p.BIOMARKERS:
    print(biomarker, ":", p.compute_cohens_d(biomarker))
print("Verification:", p.check_cohens_d())

Oldpeak : 0.8863573136001801
MaxHR : -0.8805208867307898
Age : 0.5905569689898478
Verification: True


In [145]:
result = p.high_risk_masks()
print("High-risk patients:", result['high_risk'])
print("ASY patients:", result['asy'])
print("Overlap:", result['overlap'])
print("High-risk but not ASY:", result['high_not_asy'])


High-risk patients: 65
ASY patients: 496
Overlap: 60
High-risk but not ASY: 5


In [146]:
beta, r2, weights = p.ols_disease_model()
print("Coefficients:", beta)
print("R-squared:", r2)
print("Standardized weights:", weights)

Coefficients: [ 1.06095884  0.15528719 -0.00624148  0.00388731]
R-squared: 0.0
Standardized weights: {'Oldpeak': np.float64(0.3330583048547537), 'MaxHR': np.float64(-0.3195241934944652), 'Age': np.float64(0.07374773301915745)}


In [147]:
pain_type_counts, counts, raw_c_corr, imputed_c_corr = p.chest_pain_audit()
print(pain_type_counts)
print(counts)
print("Raw correlation:", raw_c_corr)
print("Imputed correlation:", imputed_c_corr)

               patients  disease_rate  zero_chol
ChestPainType                                   
ASY                 496      0.790323        126
ATA                 173      0.138728          7
NAP                 202      0.351485         33
TA                   46      0.434783          5
ChestPainType
ASY    496
NAP    202
ATA    173
TA      46
Name: count, dtype: int64
Raw correlation: -0.2314785663766562
Imputed correlation: 0.07626169199449678


In [148]:
import importlib
import src.analyzer

importlib.reload(src.analyzer)

p = src.analyzer.CardioRiskProfiler()
p.load_from_csv(file_path)

In [149]:
print(p.prevalence_pivot().round(3))

hr_range     Low  Normal   High    All
age_range                             
40-50      0.667   0.500  0.242  0.403
50-60      0.773   0.610  0.333  0.566
60+        0.853   0.712  0.532  0.731
All        0.793   0.616  0.335  0.575


In [150]:
result = p.top_risk_profiles()
print(result)

                Biomarker  value  severity
2224                MaxHR   60.0  3.016886
2204                MaxHR   63.0  2.899023
2236                MaxHR   67.0  2.741872
2329                MaxHR   69.0  2.663297
2233                MaxHR   70.0  2.624009
3600              Oldpeak    6.2  4.982571
3521              Oldpeak    5.6  4.419919
2917              Oldpeak    5.0  3.857266
3650              Oldpeak    4.4  3.294614
3452              Oldpeak    4.2  3.107063
241             RestingBP  200.0  3.749826
365             RestingBP  200.0  3.749826
399             RestingBP  200.0  3.749826
731             RestingBP  200.0  3.749826
758             RestingBP  192.0  3.305132
1066  cholesterol_imputed  603.0  6.737497
1532  cholesterol_imputed  564.0  6.007173
993   cholesterol_imputed  529.0  5.351753
947   cholesterol_imputed  518.0  5.145764
1167  cholesterol_imputed  491.0  4.640155
